# Landmark → latent (`FFM_2_4_Landmark2Z`)

Regress the FFM-OSLO latent `z` from SYMARE 3-D landmarks. Interfaces follow
`FFM_2_3_MultiDB.ipynb` — same `DATASETS`, same `split_v21.json`, same model classes
(imported from that notebook rather than copied, so they cannot drift).

**Order of work is cheapest-first and must not be skipped:**

| cell | question | needs training? |
|---|---|---|
| 5 | Is the latent's side direction still 71.7% of variance in v2.3? | no |
| 6 | **Is there any signal at all?** `dCor²ᵤ` + permutation *p* | **no** |
| 7 | *Which frequency band* carries it | no |
| 8 | How much is usable — RFF + ridge, closed form | no SGD |
| 9 | MLP — **gated**, only runs if 6–8 pass | yes |
| 10 | End-to-end LSD through the frozen decoder, vs the mean baseline | no |

Stopping rule agreed in advance: **if cross-validated ridge R² < ~0.1, stop adding
capacity and write the negative result.**

---

## The split problem, and why it is smaller than it looks

Of the 52 high-confidence mapped subjects, the CARLab partition in `split_v21.json`
puts them:

| tier set | AE-train | AE-val | AE-test |
|---|---:|---:|---:|
| A1+A2 (52) | **41** | 3 | 8 |
| all resolved (57) | 46 | 3 | 8 |

So ~79% of the mapped subjects had their HRTFs inside the autoencoder's training set.
Three things follow, and they are not the same thing:

1. **The go/no-go metric does not care.** `dCor²ᵤ` and its permutation *p* fit no model
   and need no split. Cell 6 is valid on all 52 regardless. Do not let the split debate
   delay it.

2. **The regressor's own split must still be clean**, and it is: cell 8 cross-validates
   over subjects, so no subject is ever in its own training fold.

3. **The remaining worry is that `z` for AE-train subjects is an in-sample latent** —
   possibly tighter or lower-noise than `z` for a genuinely new subject. This is
   measurable, not arguable, and cell 5 measures it. Prior evidence says it is small:
   the amortisation gap is 0.072 dB (5.1%), cold−warm start is 0.005 dB, and test agreed
   with val within noise on every partition.

**This notebook therefore reports two numbers side by side, always:**

- `CV-all` — K-fold over all 52. Full statistical power, the headline.
- `AE-heldout` — fit on the 41 AE-train subjects, evaluate on the 11 AE-val+test ones.
  Zero in-sample-latent contamination, but N=11.

If they agree within their error bars, quote `CV-all` and disclose `AE-heldout` in one
sentence. If they disagree, the in-sample-latent effect is real and `AE-heldout` is the
honest number — with N=11 you would then be writing a negative result anyway, which is
the planned outcome.

**Do not re-split CARLab.** That would void the frozen test list and destroy the only
clean single-vs-three-database comparison in the project.

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, csv, hashlib, itertools, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
else:
    DRIVE = os.environ.get('ELEC5021_DRIVE', os.path.expanduser('~/Documents/26S2/ELEC5021'))

!pip install healpy dcor -q

PATHS = dict(
    landmarks = f'{DRIVE}/ELEC5021/SYMARE/3DShapeData.mat',
    mapping   = f'{DRIVE}/ELEC5021/analysis/symare61_to_carlab_RESOLVED.csv',
    ckpt      = f'{DRIVE}/HRTF_Checkpoints/FFM23_best.pth',
    split     = f'{DRIVE}/HRTF_Checkpoints/split_v21.json',
    ffm_nb    = f'{DRIVE}/ELEC5021/FFM_2_3_MultiDB.ipynb',   # 模型定义从这里 exec，不复制
    # 可选：缺了 cell 4 会现场生成。候选顺序和 FFM_2_3_MultiDB.ipynb 保持一致。
    mirror    = next((p for p in (
        f'{DRIVE}/ELEC5021/analysis/healpix_mirror_idx.npy',   # ← 现在的位置
        f'{DRIVE}/HRTF_Checkpoints/healpix_mirror_idx.npy',    # healpix_mirror.py 的旧输出
        f'{DRIVE}/healpix_mirror_idx.npy')                     # 更早的根目录版
        if os.path.exists(p)), f'{DRIVE}/ELEC5021/analysis/healpix_mirror_idx.npy'),
)
OPTIONAL = {'mirror'}
for k, p in PATHS.items():
    tag = 'OK ' if os.path.exists(p) else ('（可选，缺）' if k in OPTIONAL else '★缺失')
    print(f'{k:10s} {tag} {p}')
_missing = [k for k, p in PATHS.items() if not os.path.exists(p) and k not in OPTIONAL]
assert not _missing, f'先把这些放到 Drive 上: {_missing}'

# CARLab 的 .npy 目录（Cell 0 of FFM_2_3 解压出来的位置）
CARLAB = dict(left  = '/content/dataset/CARLab_44k1_left',
              right = '/content/dataset/CARLab_44k1_right',
              suffix = ('_left_full.npy', '_right_full.npy'))
if not os.path.isdir(CARLAB['left']):
    import zipfile
    os.makedirs('/content/dataset', exist_ok=True)
    for z in ('CARLab_44k1_left.zip', 'CARLab_44k1_right.zip'):
        zipfile.ZipFile(f'{DRIVE}/ELEC5021/CARLab/{z}').extractall('/content/dataset')
print('CARLab npy:', len(os.listdir(CARLAB['left'])), 'left /',
      len(os.listdir(CARLAB['right'])), 'right')

In [ ]:
# ==========================================================================
# Cell 1 — ★ 配置（唯一要动的地方）
# ==========================================================================
CFG = dict(
    # --- 用哪些 tier ---
    TIERS        = ['A1', 'A2'],     # 推荐默认 = 52 人。加 'B' -> 54。'C' 只靠姓名，别进主结果
    # --- landmark 特征 ---
    FEATURE      = 'ipsi',           # 'ipsi'(19点×3=57维) | 'all'(33×3=99) | 'dist'(同侧耳两两距离78维)
    CENTER       = 'population',     # 'population' 减去训练折的均值形状 | 'subject' 各自减质心 | 'none'
    SCALE_NORM   = False,            # ★ 默认 False：头/耳尺寸是真信号（Craig 的立场），不要归一化掉
    # --- 目标 z ---
    K_VAR        = 0.95,             # 目标 PCA 保留方差
    PROJECT_SIDE = 'auto',           # 'auto' 按实测决定 | True 强制投影掉 | False 不投影
    # --- RFF ---
    RFF_M        = 512,              # 随机特征个数
    SIGMA_GRID   = [0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0],
    # --- 评估 ---
    N_FOLDS      = 10,
    N_PERM       = 999,
    SEED         = 0,
)
print(json.dumps(CFG, indent=1))

In [ ]:
# ==========================================================================
# Cell 2 — landmarks + 映射表 -> 受试者清单
#
# 3DShapeData.mat 结构（已核对）：
#   A     (1,61) cell，每个 (33,3) float64，单位 mm，HA01..HA61 顺序
#   T_MS  (33,3) 模板 A ；T_SS (33,3) 模板 B  —— 本 notebook 不使用，仅暴露出来
#
# 33 个点的分块（由 y 坐标实测确认）：
#   0..12   一侧耳廓 13 点   (y ≈ +79 mm)
#   13..17  中矢面 5 点      (|y| ≈ 2.6 mm：鼻尖/头顶/后枕/前额/下颏一类)
#   18..30  另一侧耳廓 13 点 (y ≈ -79 mm)，与 0..12 一一对应（i <-> i+18）
#   31,32   一对左右对称点   (y = -31.8 / +35.1，肩部一类)
#
# ⚠️ 哪一侧是「左」无法从几何本身判定。见下面的 Y_POSITIVE_IS_LEFT。
# ==========================================================================
import numpy as np, scipy.io as sio

mat = sio.loadmat(PATHS['landmarks'], struct_as_record=False, squeeze_me=True)
LM_ALL = np.stack(list(mat['A'])).astype(np.float64)         # (61, 33, 3)
T_MS, T_SS = mat['T_MS'], mat['T_SS']
assert LM_ALL.shape == (61, 33, 3) and not np.isnan(LM_ALL).any()

EAR_P  = list(range(0, 13))     # y>0 侧耳
MIDSAG = list(range(13, 18))
EAR_N  = list(range(18, 31))    # y<0 侧耳
PAIR_N, PAIR_P = 31, 32

# ✅ 已用声学实测判定（见 cell 2b），不是假设：
#    在 SYMARE 的 Matlab sph2cart 坐标系里 hL 的能量集中在 y>0，且 18/18 个受试者
#    的 hL 主方向都指向 landmark 的 0..12 块。SYMARE readme 没有记载这一点。
Y_POSITIVE_IS_LEFT = True

_y = LM_ALL[..., 1]
print(f'块 y 均值:  ear_p {_y[:, EAR_P].mean():+.1f} | midsag {_y[:, MIDSAG].mean():+.1f} '
      f'| ear_n {_y[:, EAR_N].mean():+.1f} | pair {_y[:, PAIR_N].mean():+.1f}/{_y[:, PAIR_P].mean():+.1f}')
_L, _R = LM_ALL[:, EAR_P], LM_ALL[:, EAR_N].copy(); _R[..., 1] *= -1
print(f'左右耳镜像后逐轴平均差: {np.abs(_L - _R).mean((0, 1)).round(2)} mm  '
      f'(应当小但非零 —— 这就是真实的左右不对称)')
_cs = np.linalg.norm(LM_ALL - LM_ALL.mean(1, keepdims=True), axis=(1, 2))
print(f'centroid size: {_cs.mean():.1f} ± {_cs.std():.1f} mm  (range {_cs.min():.0f}–{_cs.max():.0f})')
print(f'质心跨被试 std: {LM_ALL.mean(1).std(0).round(2)} mm -> 已在公共坐标系，不需要 Procrustes')

# ---------------------------- 映射表 ----------------------------
rows = [r for r in csv.DictReader(open(PATHS['mapping'])) if r['SYMARE61']]
SPLIT = json.load(open(PATHS['split']))
where = {s: p for p in ('train', 'val', 'test') for s in SPLIT['CARLab'][p]}

SUBJ = []
for r in rows:
    if not r['final_CL'] or r['tier'] not in CFG['TIERS']:
        continue
    ha = int(r['SYMARE61'].replace('SYM_HA', ''))
    SUBJ.append(dict(ha=ha, sym=r['SYMARE61'], cl=r['final_CL'], tier=r['tier'],
                     r=float(r['r']), ae_part=where[r['final_CL']]))
SUBJ.sort(key=lambda s: s['ha'])
assert len({s['cl'] for s in SUBJ}) == len(SUBJ), '映射不是单射 —— 停下来查'

from collections import Counter
print(f"\n纳入 {len(SUBJ)} 人  tier={Counter(s['tier'] for s in SUBJ)}")
print(f"  AE 分区: {dict(Counter(s['ae_part'] for s in SUBJ))}")
AE_HELDOUT = [s for s in SUBJ if s['ae_part'] in ('val', 'test')]
print(f"  AE-heldout 子集 = {len(AE_HELDOUT)} 人 -> {2*len(AE_HELDOUT)} 只耳（cell 8 的稳健性检查）")

In [ ]:
# ==========================================================================
# Cell 2b — ★ 左右手别的声学判定（SYMARE readme 对 landmark 只字未提）
#
# 思路：landmark 和 HRIR 是同一批受试者、同一个 SYMARE 坐标系。
#   1. azim/elev 经 Matlab sph2cart（右手系）得到 393 个方向的单位向量 D
#   2. 每个受试者取 hL / hR 的宽带能量作权重，算出「同侧主方向」dL / dR
#   3. 和 landmark 两个耳块的方向 ê_P / ê_N 做内积
#   dL·ê_P > dL·ê_N 就说明 0..12 块是左耳。
#
# 实测结果（HA01,02,03,05,07,09,11,13,15,20,25,30,35,40,45,50,55,61）：
#   hL 能量加权 <y> = +0.495 ± 0.031，hR = -0.494 ± 0.034
#   y>0 半球上的 ILD(hL vs hR) = +12.34 ± 2.03 dB，18/18 为正
#   dL·ê_P 在 +0.74..+0.93 之间，dL·ê_N 恒为其相反数 —— 18/18 判定「P = 左耳」
# 有 SYMARE/HRIRs 就重跑一遍；没有就跳过（结论已 hard-code 在上面）。
# ==========================================================================
SYM_HRIR = f'{DRIVE}/ELEC5021/SYMARE/HRIRs'
if os.path.isdir(f'{SYM_HRIR}/Acoustic'):
    import glob, re
    _a = sio.loadmat(f'{SYM_HRIR}/Parameters/azim.mat')
    _e = sio.loadmat(f'{SYM_HRIR}/Parameters/elev.mat')
    _az = _a[[k for k in _a if not k.startswith('__')][0]].ravel()
    _el = _e[[k for k in _e if not k.startswith('__')][0]].ravel()
    D = np.stack([np.cos(_el)*np.cos(_az), np.cos(_el)*np.sin(_az), np.sin(_el)], 1)
    ok = tot = 0
    for f in sorted(glob.glob(f'{SYM_HRIR}/Acoustic/HA*.mat')):
        k = int(re.search(r'HA(\d+)', f).group(1)); P = LM_ALL[k-1]
        cP, cN = P[EAR_P].mean(0), P[EAR_N].mean(0); mid = (cP+cN)/2
        eP, eN = (cP-mid)/np.linalg.norm(cP-mid), (cN-mid)/np.linalg.norm(cN-mid)
        m = sio.loadmat(f); wL = (m['hL']**2).sum(0); wR = (m['hR']**2).sum(0)
        dL = (D*(wL/wL.sum())[:, None]).sum(0); dL /= np.linalg.norm(dL)
        dR = (D*(wR/wR.sum())[:, None]).sum(0); dR /= np.linalg.norm(dR)
        ok += (dL @ eP > dL @ eN) and (dR @ eN > dR @ eP); tot += 1
    print(f'声学判定: {ok}/{tot} 个受试者支持「landmark 块 0..12 = 左耳」')
    assert ok == tot, ('★ 判定不一致 —— 停下来。要么 Y_POSITIVE_IS_LEFT 该翻，'
                       '要么 landmark 和 HRIR 不在同一个坐标系里。')
    assert Y_POSITIVE_IS_LEFT, 'Y_POSITIVE_IS_LEFT 和声学判定矛盾'
else:
    print(f'[跳过] 没找到 {SYM_HRIR}/Acoustic —— 沿用已验证的 Y_POSITIVE_IS_LEFT = True')
    print('       该结论来自 18/18 受试者的声学判定，不是假设。')

In [ ]:
# ==========================================================================
# Cell 3 — landmark 特征（每只耳一行，右耳镜像进「左耳坐标系」）
#
# 和 MonauralHRTFDataset 完全同一个约定：HRTF 右耳沿中矢面镜像，
# 所以 landmark 也必须镜像 —— 否则特征和目标不在同一个参考系里。
# 镜像 = y 取反 + 交换两个耳块 + 交换那对对称点。
# ==========================================================================
def mirror_config(P):
    """P: (33,3) -> 镜像后的 (33,3)，使「y>0 那一侧」变成原来的另一侧"""
    Q = P.copy(); Q[:, 1] *= -1
    Q[EAR_P], Q[EAR_N] = P[EAR_N] * [1, -1, 1], P[EAR_P] * [1, -1, 1]
    Q[PAIR_P], Q[PAIR_N] = P[PAIR_N] * [1, -1, 1], P[PAIR_P] * [1, -1, 1]
    return Q

IPSI_IDX = EAR_P + MIDSAG + [PAIR_P]          # 19 点：同侧耳 + 中矢面 + 同侧肩

def featurise(P):
    """P 已在左耳坐标系（同侧耳 = EAR_P）"""
    if CFG['FEATURE'] == 'all':
        X = P.reshape(-1)
    elif CFG['FEATURE'] == 'ipsi':
        X = P[IPSI_IDX].reshape(-1)
    elif CFG['FEATURE'] == 'dist':
        E = P[EAR_P]
        X = np.array([np.linalg.norm(E[i] - E[j])
                      for i, j in itertools.combinations(range(len(EAR_P)), 2)])
    else:
        raise ValueError(CFG['FEATURE'])
    return X.astype(np.float64)

EARS, L_list = [], []
for s in SUBJ:
    P0 = LM_ALL[s['ha'] - 1]
    ipsi_left = P0 if Y_POSITIVE_IS_LEFT else mirror_config(P0)
    for side, P in (('left', ipsi_left), ('right', mirror_config(ipsi_left))):
        if CFG['CENTER'] == 'subject':
            P = P - P.mean(0)
        EARS.append(dict(**s, side=side, key=f"{s['cl']}_{side}"))
        L_list.append(featurise(P))
L_raw = np.stack(L_list)                                   # (2N, d)
side_is_right = np.array([e['side'] == 'right' for e in EARS])

print(f'L_raw {L_raw.shape}  feature={CFG["FEATURE"]}  center={CFG["CENTER"]}')
_d = np.abs(L_raw[~side_is_right] - L_raw[side_is_right]).mean()
print(f'左耳特征 vs 右耳特征 平均 |差| = {_d:.2f} mm')
assert 0.2 < _d < 15, ('镜像疑似出错：完全为 0 说明两耳被当成同一个东西，'
                       '过大说明块索引或镜像写错了')
if CFG['SCALE_NORM']:
    print('⚠️ SCALE_NORM=True：正在把头/耳尺寸归一化掉。Craig 的立场是尺寸就是信号，'
          '这一项会削掉最可预测的成分。确认你真的要这么做。')

In [ ]:
# ==========================================================================
# Cell 4 — 载入 AE，编码出 z
#   模型类不复制，直接 exec FFM_2_3_MultiDB.ipynb 里定义它们的那个 cell。
# ==========================================================================
import torch, healpy as hp, nbformat

_nb = nbformat.read(PATHS['ffm_nb'], as_version=4)
_src = [''.join(c['source']) for c in _nb.cells
        if c['cell_type'] == 'code' and 'class FFM_OSLO_Autoencoder' in ''.join(c['source'])]
assert len(_src) == 1, f'在 FFM_2_3_MultiDB.ipynb 里找到 {len(_src)} 个模型定义 cell，应当恰好 1 个'
exec(_src[0], globals())
print('模型类已从 FFM_2_3_MultiDB.ipynb 导入（未复制，不会漂移）')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
n_ds, n_bins = ckpt['model_state_dict']['decoder.film.beta'].shape
print(f"checkpoint: epoch {ckpt['epoch']+1} | val RMSE {ckpt['val_rmse']:.4f} dB | "
      f'{n_ds} 库 / {n_bins} bin')
assert n_bins == 93, f'num_bins={n_bins}，不是 93 —— freq_cap 改过了，先搞清楚'

model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                             num_datasets=n_ds, num_bins=n_bins,
                             tie_encoder_film=True).to(device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()

# checkpoint 里嵌的 split 才是权威的
assert ckpt['split']['CARLab']['train'] == SPLIT['CARLab']['train'], \
    'split_v21.json 和 checkpoint 里的划分不一致 —— 以 checkpoint 为准，停下来查'

# 镜像置换：Drive 上有就用，没有就现场生成 —— 两者已核对为 bit-for-bit 相同
if os.path.exists(PATHS['mirror']):
    MIRROR = np.load(PATHS['mirror'])
    print('mirror <- Drive')
else:
    _npix = hp.nside2npix(16)
    _th, _ph = hp.pix2ang(16, np.arange(_npix))
    MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
    print('mirror 现场生成（与 Drive 上那份已核对为完全一致）')
assert len(np.unique(MIRROR)) == len(MIRROR), '镜像不是双射'
assert np.array_equal(MIRROR[MIRROR], np.arange(len(MIRROR))), '镜像不是对合'

CARLAB_DID = 0                                    # FiLM 行号，见 FFM_2_3 的 DATASETS

# ★ encoder.forward 只收 x —— 数据集条件化是【编码前的逆 FiLM】，做在
#   FFM_OSLO_Autoencoder.forward 里。直接 model.encoder(x) 会把测量染色
#   一起编进 z，正是模型 docstring 警告的那件事。这里复刻同一条路径。
def encode(x_t, did_t):
    xn = model.decoder.film.inverse(x_t, did_t) if model.tie_encoder_film else x_t
    return model.encoder(xn)

def load_ear(cl, side):
    sfx = CARLAB['suffix'][0 if side == 'left' else 1]
    x = np.load(os.path.join(CARLAB[side], f'{cl}{sfx}')).astype(np.float32)[:n_bins]
    return x[:, MIRROR] if side == 'right' else x

Z_list = []
with torch.no_grad():
    did = torch.tensor([CARLAB_DID], device=device)
    for j, e in enumerate(EARS):
        xt = torch.from_numpy(load_ear(e['cl'], e['side']))[None].to(device)
        z = encode(xt, did)
        if j == 0:                       # 和整模型的 forward 对一次，必须逐位相同
            z_ref = model(xt, did)[1]
            assert torch.equal(z, z_ref), 'encode() 和 model.forward 的 latent 不一致'
            print('encode() 与 model.forward 的 z 逐位一致 ✓')
        Z_list.append(z[0].float().cpu().numpy())
Z_fd = np.stack(Z_list)                                        # (2N, 93, 64)
Z = Z_fd.reshape(len(Z_fd), -1)                                # (2N, 5952)
print(f'Z_fd {Z_fd.shape} -> Z {Z.shape}')
DF = ckpt.get('config', {}).get('sample_rate', 44100) / 256
freqs = np.arange(n_bins) * DF
print(f'df = {DF:.2f} Hz, 频段 0–{freqs[-1]/1000:.2f} kHz')
del ckpt

In [ ]:
# ==========================================================================
# Cell 5 — latent 诊断（先测，别照抄 v1 的数字）
#
#   v1 的方差分解是 71.7% = 「哪只耳」。但 FFM_2_3 的验收测试写着：
#   v2.3 镜像生效后这一项【应当降到接近 0】。所以必须现测，不能假设。
# ==========================================================================
v = Z[side_is_right].mean(0) - Z[~side_is_right].mean(0)
v = v / np.linalg.norm(v)
Z_nos = Z - np.outer(Z @ v, v)
side_frac = 1 - Z_nos.var() / Z.var()
print(f'① side 方向占 latent 方差: {side_frac:.1%}   (v1 = 71.7%；v2.3 若镜像生效应接近 0)')

do_proj = side_frac > 0.05 if CFG['PROJECT_SIDE'] == 'auto' else bool(CFG['PROJECT_SIDE'])
Z_use = Z_nos if do_proj else Z
print(f'   -> {"投影掉 side 方向" if do_proj else "不投影（side 已经很小）"}')

# ② in-sample latent 有没有系统性差异 —— 这是整个 split 争议里唯一能测的部分
tr_m = np.array([e['ae_part'] == 'train' for e in EARS])
for nm, M in (('AE-train', tr_m), ('AE-heldout', ~tr_m)):
    print(f'② {nm:11s} n={M.sum():3d} ears | ||z|| {np.linalg.norm(Z_use[M],axis=1).mean():8.2f}'
          f' ± {np.linalg.norm(Z_use[M],axis=1).std():6.2f} | var {Z_use[M].var():.4f}')
from scipy import stats
# ★ 必须在【人】层面做：左右耳镜像后相关 0.978，按耳做会把 N 虚增一倍、p 虚低一半
_nrm = lambda M: np.linalg.norm(Z_use[M], axis=1).reshape(-1, 2).mean(1)
_t  = stats.ttest_ind(_nrm(tr_m), _nrm(~tr_m))
_te = stats.ttest_ind(np.linalg.norm(Z_use[tr_m], axis=1), np.linalg.norm(Z_use[~tr_m], axis=1))
print(f'   ||z|| 两组差异（★人层面 n={len(_nrm(tr_m))} vs {len(_nrm(~tr_m))}）'
      f' t={_t.statistic:+.2f}, p={_t.pvalue:.3f}')
print(f'   （耳层面 t={_te.statistic:+.2f}, p={_te.pvalue:.3f} —— 仅供对照，左右耳不独立，不要引用）')
print(f'   -> {"没有可测差异，CV-all 可用" if _t.pvalue > .05 else "★有差异，以 AE-heldout 为准"}')
print('   注意：这只是 ||z|| 一个一维汇总量，是很弱的代理。'
      'cell 8 的 AE-heldout R² 才是对这个顾虑的直接检验。')

# ③ 有效维度
Zc = Z_use - Z_use.mean(0)
S = np.linalg.svd(Zc, compute_uv=False)
cum = np.cumsum(S**2) / (S**2).sum()
K = int(np.searchsorted(cum, CFG['K_VAR']) + 1)
print(f'③ 保留 {CFG["K_VAR"]:.0%} 方差需要 K = {K} 维 '
      f'(上限 = min(2N-1, 5952) = {min(len(Z_use)-1, Z.shape[1])})')
print('   ★ 与 analysis/FFM_2_1_latent_structure.py 交叉核对；那个脚本至今没跑过。')

U, _, _ = np.linalg.svd(Zc, full_matrices=False)
Z_pca = U[:, :K] * np.sqrt(len(Zc))
print(f'   Z_pca {Z_pca.shape}')

In [ ]:
# ==========================================================================
# Cell 6 — ★★ 去/留判据：dCor²ᵤ + 置换检验（不训练任何模型）
#
#   这一格【不受 split 争议影响】：它不拟合模型，不需要划分。
#   零假设分布只取决于 N：N=57 -> 95 分位 0.040；N=115 -> 0.021。
#   ⚠️ 不要用 dcor.distance_correlation（有偏）也不要用 CKA：
#      N=57、独立数据上它们分别读 0.830 和 0.378。
# ==========================================================================
import dcor
rng = np.random.default_rng(CFG['SEED'])

def whiten(X, keep=0.95):
    X = X - X.mean(0)
    U, S, _ = np.linalg.svd(X, full_matrices=False)
    k = int(np.searchsorted(np.cumsum(S**2) / (S**2).sum(), keep) + 1)
    return U[:, :k] * np.sqrt(len(X)), k

def dcor_test(L, Z_, nperm, tag=''):
    obs = dcor.u_distance_correlation_sqr(L, Z_)
    null = np.array([dcor.u_distance_correlation_sqr(L, Z_[rng.permutation(len(Z_))])
                     for _ in range(nperm)])
    p = (1 + (null >= obs).sum()) / (1 + nperm)
    print(f'  {tag:22s} dCor²ᵤ = {obs:+.4f} | 零分布 95分位 {np.quantile(null,.95):+.4f} '
          f'| p = {p:.4f} {"✅" if p < .05 else "—"}')
    return obs, p

Lw, kL = whiten(L_raw)
Zw, kZ = whiten(Z_use)
print(f'白化后维度: landmarks {kL} | latent {kZ} | N = {len(Lw)} ears '
      f'({len(SUBJ)} subjects)\n')

print('每只耳一行（N 是耳数，但左右耳不独立 —— 这个 p 偏乐观，只做参考）:')
dcor_test(Lw, Zw, CFG['N_PERM'], 'ear-level')

# 主结果：把同一个人的左右耳平均成一行，N = 人数，样本独立
def to_subject(M):
    out = []
    for i in range(0, len(M), 2):
        out.append(M[i:i+2].mean(0))
    return np.stack(out)
Ls, _ = whiten(to_subject(L_raw))
Zs, _ = whiten(to_subject(Z_use))
print(f'\n★ 主结果 —— 每人一行，N = {len(Ls)}（独立样本）:')
OBS, PVAL = dcor_test(Ls, Zs, CFG['N_PERM'], 'subject-level')

print('\n负对照（应当都不显著）:')
dcor_test(Ls, Zs[rng.permutation(len(Zs))], CFG['N_PERM'], 'shuffled z')
dcor_test(rng.normal(size=Ls.shape), Zs, CFG['N_PERM'], 'random features')

print(f'\n判据: {"有信号，继续 cell 7" if PVAL < .05 else "★ 没有可测信号 —— 这本身就是结果，去写 cell 10 的负结果"}')
print('注意 dCor² 会饱和：真实共享方差 10%→70% 只让它从 0.42 走到 0.76。')
print('它回答「有没有」，不回答「有多少」。「多少」看 cell 8 的 CV-R²。')

In [ ]:
# ==========================================================================
# Cell 7 — 信号在哪个频段（这才是科学上有意思的输出，不是那个单一分数）
# ==========================================================================
EDGES = ((0, 500), (500, 6000), (6000, 12000), (12000, int(freqs[-1]) + 1))
Zf_sub = to_subject(Z_fd.reshape(len(Z_fd), -1)).reshape(-1, n_bins, 64)

print(f'{"band":16s}{"bins":>6}{"dCor²ᵤ":>10}{"p":>8}')
band_res = {}
for lo, hi in EDGES:
    m = (freqs >= lo) & (freqs < hi)
    Zb, _ = whiten(Zf_sub[:, m, :].reshape(len(Zf_sub), -1))
    o, p = dcor.u_distance_correlation_sqr(Ls, Zb), None
    null = np.array([dcor.u_distance_correlation_sqr(Ls, Zb[rng.permutation(len(Zb))])
                     for _ in range(199)])
    p = (1 + (null >= o).sum()) / 200
    band_res[(lo, hi)] = (o, p)
    print(f'{f"{lo/1000:.1f}–{hi/1000:.1f} kHz":16s}{int(m.sum()):>6}{o:>10.4f}{p:>8.3f}'
          f' {"✅" if p < .05 else ""}')
print('\n对照：重建残差/基线 的分带比值是 DC<500Hz 59.3%(最差) / 6–12kHz 25.3%(最好)。')
print('如果几何的信号恰好落在重建最好的 notch 带，那是好消息；落在低频就只是头尺寸。')

In [ ]:
# ==========================================================================
# Cell 8 — RFF + ridge（闭式解，不做 SGD）
#
#   d = 57~99 维时，random Fourier features 就是 Rahimi–Recht 随机特征，
#   γ(a)·γ(b) ≈ exp(-2π²σ²‖a-b‖²)，所以 RFF+ridge ≈ RBF 核岭回归。
#   σ 是核带宽，不是「让网络学高频」。N≈52 时这是低方差估计量，正是想要的。
#
#   ★ 标准化统计量只从训练折算 —— 否则 σ 的含义随折变化，且泄漏。
# ==========================================================================
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

L_sub = to_subject(L_raw)
Z_sub = to_subject(Z_use)
Zc_sub = Z_sub - Z_sub.mean(0)
_U, _S, _Vt = np.linalg.svd(Zc_sub, full_matrices=False)
_k = int(np.searchsorted(np.cumsum(_S**2)/(_S**2).sum(), CFG['K_VAR']) + 1)
Y = _U[:, :_k] * _S[:_k]                       # 目标 = 人层面 latent 的主成分
# 逆变换（cell 10 要用）：Zc ≈ Y @ _Vt[:_k]
assert np.abs(Y @ _Vt[:_k] - Zc_sub).max() < 1e-6 * np.abs(Zc_sub).max() + 1e-3 or _k < len(_S), 'PCA 逆变换核对'
part = np.array([s['ae_part'] for s in SUBJ])
print(f'X {L_sub.shape} -> Y {Y.shape} | N = {len(Y)} 人')

def rff(X, B, b):
    return np.sqrt(2.0 / B.shape[1]) * np.cos(X @ B + b)

def fit_eval(idx_tr, idx_te, sigma, alpha, seed=0):
    mu, sd = L_sub[idx_tr].mean(0), L_sub[idx_tr].std(0) + 1e-9
    Xtr, Xte = (L_sub[idx_tr] - mu) / sd, (L_sub[idx_te] - mu) / sd
    r = np.random.default_rng(seed)
    B = r.normal(scale=sigma, size=(Xtr.shape[1], CFG['RFF_M']))
    b = r.uniform(0, 2 * np.pi, CFG['RFF_M'])
    m = Ridge(alpha=alpha).fit(rff(Xtr, B, b), Y[idx_tr])
    pred = m.predict(rff(Xte, B, b))
    ss_res = ((Y[idx_te] - pred) ** 2).sum()
    ss_tot = ((Y[idx_te] - Y[idx_tr].mean(0)) ** 2).sum()   # ★ 基线 = 训练折均值
    return 1 - ss_res / ss_tot, pred

ALPHAS = [1e-1, 1, 10, 100, 1e3, 1e4]
def cv_r2(idx, folds, tag):
    best = (-9, None, None)
    for sg in CFG['SIGMA_GRID']:
        for al in ALPHAS:
            kf = KFold(folds, shuffle=True, random_state=CFG['SEED'])
            sc = [fit_eval(idx[a], idx[b], sg, al)[0] for a, b in kf.split(idx)]
            if np.mean(sc) > best[0]:
                best = (np.mean(sc), sg, al, np.std(sc) / np.sqrt(folds))
    print(f'  {tag:14s} CV-R² = {best[0]:+.4f} ± {best[3]:.4f} (SEM)  '
          f'[σ={best[1]}, α={best[2]:g}]')
    return best

print('\n★ 两个数并排报，见 notebook 开头的说明:')
idx_all = np.arange(len(Y))
R2_ALL = cv_r2(idx_all, CFG['N_FOLDS'], 'CV-all')

idx_tr = np.where(part == 'train')[0]
idx_ho = np.where(part != 'train')[0]
best_s, best_a, best_v = None, None, -9
for sg in CFG['SIGMA_GRID']:
    for al in ALPHAS:
        kf = KFold(5, shuffle=True, random_state=CFG['SEED'])
        v_ = np.mean([fit_eval(idx_tr[a], idx_tr[b], sg, al)[0] for a, b in kf.split(idx_tr)])
        if v_ > best_v: best_v, best_s, best_a = v_, sg, al
r2_ho, _ = fit_eval(idx_tr, idx_ho, best_s, best_a)
print(f'  {"AE-heldout":14s} R²   = {r2_ho:+.4f}            [σ={best_s}, α={best_a:g}] '
      f'fit on {len(idx_tr)}, test on {len(idx_ho)} 人')

print('\n负对照:')
perm = np.random.default_rng(1).permutation(len(Y))
_Ysave = Y.copy(); Y = Y[perm]
cv_r2(idx_all, CFG['N_FOLDS'], 'shuffled Y')
Y = _Ysave

print(f'\n停止规则: CV-R² {"≥" if R2_ALL[0] >= .1 else "<"} 0.1 '
      f'-> {"可以进 cell 9 试 MLP" if R2_ALL[0] >= .1 else "★ 停。不要加容量，去写负结果。"}')

In [ ]:
# ==========================================================================
# Cell 9 — MLP（★ 有闸门：cell 8 不过就不跑）
# ==========================================================================
GATE = (PVAL < 0.05) and (R2_ALL[0] >= 0.10)
if not GATE:
    print(f'闸门关闭 (dCor p={PVAL:.4f}, CV-R²={R2_ALL[0]:+.4f})。')
    print('按事先约定的停止规则，不加容量。直接去 cell 10 出负结果的端到端数字。')
    print('要强行跑就把 GATE 改成 True —— 但那样的结果在 N=%d 上没有说服力。' % len(Y))
else:
    import torch.nn as nn
    print('闸门开启，跑 MLP。仍然是 KFold，仍然只报 CV。')
    def run_fold(a, b, h=128, epochs=400, lr=1e-3, wd=1e-2, seed=0):
        torch.manual_seed(seed)
        mu, sd = L_sub[a].mean(0), L_sub[a].std(0) + 1e-9
        Xa = torch.tensor((L_sub[a]-mu)/sd, dtype=torch.float32)
        Xb = torch.tensor((L_sub[b]-mu)/sd, dtype=torch.float32)
        Ya = torch.tensor(Y[a], dtype=torch.float32)
        net = nn.Sequential(nn.Linear(Xa.shape[1], h), nn.GELU(), nn.Dropout(0.2),
                            nn.Linear(h, h), nn.GELU(), nn.Linear(h, Y.shape[1]))
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
        for _ in range(epochs):
            opt.zero_grad(); nn.functional.mse_loss(net(Xa), Ya).backward(); opt.step()
        net.eval()
        with torch.no_grad(): pr = net(Xb).numpy()
        return 1 - ((Y[b]-pr)**2).sum() / ((Y[b]-Y[a].mean(0))**2).sum()
    kf = KFold(CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])
    sc = [run_fold(a, b) for a, b in kf.split(idx_all)]
    print(f'  MLP CV-R² = {np.mean(sc):+.4f} ± {np.std(sc)/np.sqrt(len(sc)):.4f}')

In [ ]:
# ==========================================================================
# Cell 10 — 端到端：ẑ 过冻结 decoder -> LSD，和均值基线比
#
#   ★ 口径必须写全，否则不可比：
#     LSD = 先沿频率 RMS，再对方向【算术】平均  （文献口径，系统性最小）
#     93 bins / DC–15.85 kHz / 44.1 kHz / dB magnitude
#   ★ 这个数【不能】和重建的 1.2909 dB 放同一张表 —— 不同任务。
# ==========================================================================
def lsd(pred, gt):
    """(B,93,3072) -> 标量。先沿频率 RMS，再对方向和样本算术平均"""
    return np.sqrt(((pred - gt) ** 2).mean(1)).mean()

# 用 AE-heldout 的人做端到端评估（他们的 HRTF 没进过 AE 训练）
ho = [s for s in SUBJ if s['ae_part'] != 'train']
print(f'端到端评估集: {len(ho)} 人 / {2*len(ho)} 只耳（AE 没见过的 CARLab 受试者）')
print('⚠️ N 很小。这是定性确认，不是可以单独引用的 test 数字。')

# 训练集均值谱基线（只用 AE-train 的 CARLab 受试者，避免泄漏）
mu_spec = np.zeros((n_bins, 3072), dtype=np.float64); cnt = 0
for sid in SPLIT['CARLab']['train']:
    for side in ('left', 'right'):
        sfx = CARLAB['suffix'][0 if side == 'left' else 1]
        if not os.path.exists(os.path.join(CARLAB[side], f'{sid}{sfx}')): continue
        mu_spec += load_ear(sid, side).astype(np.float64); cnt += 1
mu_spec /= cnt
print(f'均值基线来自 {cnt} 只训练耳')

# 用 cell 8 选中的 σ/α 在 AE-train 上重拟合，预测 heldout 的 ẑ
# pred_pcs 的行序 = idx_ho 的顺序；逆 PCA 回到 5952 维再 reshape 成 (93,64)
_, pred_pcs = fit_eval(idx_tr, idx_ho, best_s, best_a)
Z_hat = pred_pcs @ _Vt[:_k] + Z_sub.mean(0)                 # (len(idx_ho), 5952)
print(f'ẑ {Z_hat.shape} -> 每人 reshape 成 ({n_bins}, 64)')

GT, PR, BL = [], [], []
with torch.no_grad():
    did = torch.tensor([CARLAB_DID], device=device)
    for row, i_sub in enumerate(idx_ho):
        cl = SUBJ[i_sub]['cl']
        zt = torch.tensor(Z_hat[row].reshape(n_bins, 64),
                          dtype=torch.float32, device=device)[None]
        # decoder(z, dataset_id) 会把正向 FiLM 加回去 —— 回到 CARLab 的原始空间
        rec = model.decoder(zt, did)[0].float().cpu().numpy()
        for side in ('left', 'right'):
            GT.append(load_ear(cl, side)); PR.append(rec); BL.append(mu_spec)
GT, PR, BL = np.stack(GT), np.stack(PR), np.stack(BL)
print('⚠️ ẑ 是【人】层面的（左右耳已平均），所以同一个 ẑ 同时对上左右耳两个 GT。')
print('   左右耳真实不对称约 1.5 mm，这一步会把它算成误差 —— 是保守的方向。')

print(f'\n{"":22s}{"LSD [dB]":>10}')
print(f'{"landmark -> ẑ -> HRTF":22s}{lsd(PR, GT):>10.3f}')
print(f'{"population mean":22s}{lsd(BL, GT):>10.3f}   ← 这才是判据')
print(f'{"reduction":22s}{100*(1-lsd(PR,GT)/lsd(BL,GT)):>9.1f}%')
print('\n口径: 93 bins, DC–15.85 kHz, 44.1 kHz, dB magnitude,')
print('      LSD = RMS-over-frequency 然后对方向和样本算术平均。')
print('★ 不要和重建的 1.2909 dB 并排 —— 不同任务、不同问题。')

In [ ]:
# ==========================================================================
# Cell 11 — 导出缓存给 FFM_2_5_MLP_train.ipynb
#   把编码好的人层面数据存下来，训练 notebook 不必每次重跑 104 只耳的 encoder。
#   同时存受试者名单的 md5 —— 名单一变，训练 notebook 会拒绝复用旧 checkpoint。
# ==========================================================================
CACHE = f'{DRIVE}/HRTF_Checkpoints/landmark2z_cache.npz'
_ids = np.array([s['cl'] for s in SUBJ])
_hash = hashlib.md5(','.join(_ids).encode()).hexdigest()[:12]
np.savez_compressed(
    CACHE,
    L_sub=L_sub, Z_sub=Z_sub,                     # (52, 57) / (52, 5952)  人层面
    cl=_ids, sym=np.array([s['sym'] for s in SUBJ]),
    tier=np.array([s['tier'] for s in SUBJ]),
    ae_part=np.array([s['ae_part'] for s in SUBJ]),
    n_bins=n_bins, latent_dim=64, freqs=freqs,
    side_frac=side_frac, projected=do_proj,
    z_mean=Z_sub.mean(0),
    feature=CFG['FEATURE'], center=CFG['CENTER'], tiers=np.array(CFG['TIERS']),
    subject_hash=_hash,
)
print(f'缓存 -> {CACHE}')
print(f'  L_sub {L_sub.shape} | Z_sub {Z_sub.shape} | {len(SUBJ)} 人 | hash {_hash}')
print(f'  ae_part: {dict(Counter(s["ae_part"] for s in SUBJ))}')
print(f'  参考基准（FFM_2_5 要打败的东西）: ridge CV-R² = {R2_ALL[0]:+.4f} ± {R2_ALL[3]:.4f}')

## 交付前的检查清单

1. ~~`Y_POSITIVE_IS_LEFT` 是否正确~~ —— **已解决**。SYMARE readme 完全没写 landmark，
   但用声学可以判定：hL 的宽带能量在 Matlab sph2cart 坐标系里集中于 y>0
   (`<y>_L = +0.495 ± 0.031`，y>0 半球 ILD `+12.34 ± 2.03 dB`，18/18 为正)，
   且 18/18 个受试者的 `dL·ê_P > dL·ê_N`。cell 2b 会重跑并大声失败。

2. **cell 5 ① 的 side 方差** —— 如果 v2.3 的镜像生效了，它应当接近 0 而不是 71.7%。
   照抄 v1 的 71.7% 是错的，`analysis/latent_variance_decomp.py` 在新 checkpoint 上
   重跑一次可以交叉核对。

3. **cell 5 ③ 的 K** —— 和 `analysis/FFM_2_1_latent_structure.py` 对一下。
   那个脚本至今没跑过，而它本该是这条线的第一步。

4. **cell 5 ② 的 t 检验** —— 这是整个 split 争议里唯一能测的东西。p > 0.05 就报
   `CV-all`；p < 0.05 就以 `AE-heldout` 为准并在论文里说明。

5. **HUTUBS 重复对的天花板** —— `pp22/pp88`（同一个人测两次）和 `pp1/pp96`（同一个
   manikin 测两次）编码后的 latent 距离，除以人群中位两两距离，是任何几何预测器的
   经验上界。没有它就无法判断 R²=0.3 到底是差还是接近天花板。那两对在 HUTUBS 里，
   需要单独一格，本 notebook 没有包含。

6. **报数纪律** —— cell 10 的 LSD 单独成表，口径写全，不与 1.2909 dB 同框。